In [2]:
#-----------------------------验证nc文件中的变量有哪些----------------时间跨度--------------------
import netCDF4 as nc
import numpy as np

file_path = r"F:\TUB\greenlandProject\ERA5LAND\ProcessFile\ERA5LANDfeature.nc"
data = nc.Dataset(file_path)

# 获取所有变量名
variables = data.variables.keys()

# 获取时间变量（假设名为'time'）
time_var = data.variables.get('time', None)
if time_var:
    time_units = time_var.units if hasattr(time_var, 'units') else 'unknown units'
    time_values = time_var[:]
    time_span = f"{min(time_values)} to {max(time_values)} ({time_units})"
else:
    time_span = "No time variable found"

print("Variables and their time spans:")
for var in variables:
    var_obj = data.variables[var]
    print(f"- {var}: {var_obj.shape}")
    
print(f"\nTime span: {time_span}")
data.close()

Variables and their time spans:
- number: ()
- valid_time: (902,)
- latitude: (351,)
- longitude: (751,)
- expver: (902,)
- t2m: (902, 351, 751)
- asn: (902, 351, 751)
- fal: (902, 351, 751)
- slhf: (902, 351, 751)
- str: (902, 351, 751)
- ssrd: (902, 351, 751)
- strd: (902, 351, 751)
- u10: (902, 351, 751)
- v10: (902, 351, 751)
- sp: (902, 351, 751)

Time span: No time variable found


In [5]:
#-----------------------------提取1950年4月所有气象参数到有LLA的文件中-----------并以年月-allfeature命名新文件---------
import pandas as pd
import xarray as xr
from tqdm import tqdm  # 用于显示进度条

# 文件路径
csv_path = r"F:\TUB\greenlandProject\ERA5LAND\LaiLonAlt.csv"
nc_path = r"F:\TUB\greenlandProject\ERA5LAND\ProcessFile\ERA5LANDfeature.nc"
output_path = r"F:\TUB\greenlandProject\ERA5LAND\195004allfeature.csv"

# 读取CSV文件
print("正在读取CSV文件...")
df = pd.read_csv(csv_path)
print(f"CSV文件读取完成，共{len(df)}行数据")

# 读取NetCDF文件
print("正在读取NetCDF文件...")
ds = xr.open_dataset(nc_path)
print("NetCDF文件读取完成")

# 选择1950年4月的数据
year = 1950
month = 4
print(f"正在选择{year}年{month}月数据...")

# 查找对应时间点
try:
    # 将valid_time转换为datetime格式以便查找
    times = pd.to_datetime(ds['valid_time'].values)
    target_time = pd.Timestamp(f"{year}-{month:02d}-01")
    
    # 找到最接近目标时间的索引
    time_idx = (times == target_time).argmax()
    if times[time_idx] != target_time:
        print(f"警告: 未找到精确匹配的时间，使用最接近的时间: {times[time_idx]}")
    
    # 选择对应时间的数据
    ds_selected = ds.isel(valid_time=time_idx)
    print(f"已选择时间: {times[time_idx]}")
    
except Exception as e:
    print("时间选择出错:", e)
    raise

# 准备存储气象数据的字典
data_dict = {
    'sp': [], 'ssrd': [], 'str': [], 'strd': [], 
    't2m': [], 'u10': [], 'v10': [], 'asn': [], 'fal': []
}

# 遍历CSV中的每个经纬度点
print("开始提取气象数据...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    lat = row['latitude']
    lon = row['longitude']
    
    try:
        # 从NetCDF中提取最近点的数据
        point_data = ds_selected.sel(
            latitude=lat, 
            longitude=lon, 
            method='nearest'
        )
        
        # 将数据添加到字典中
        for var in data_dict.keys():
            data_dict[var].append(float(point_data[var].values))
            
    except Exception as e:
        print(f"\n处理经纬度({lat}, {lon})时出错:", e)
        # 如果出错，添加NaN值
        for var in data_dict.keys():
            data_dict[var].append(float('nan'))

# 将气象数据添加到DataFrame中
for var in data_dict.keys():
    df[var] = data_dict[var]

# 添加年份和月份列
df['year'] = year
df['month'] = month

# 保存结果到新的CSV文件
print("正在保存结果...")
df.to_csv(output_path, index=False)

print(f"\n处理完成，结果已保存到 {output_path}")
print(f"总处理点数: {len(df)}")
print("新增列:", list(data_dict.keys()) + ['year', 'month'])

正在读取CSV文件...
CSV文件读取完成，共86629行数据
正在读取NetCDF文件...
NetCDF文件读取完成
正在选择1950年4月数据...
已选择时间: 1950-04-01 00:00:00
开始提取气象数据...


100%|██████████| 86629/86629 [08:41<00:00, 166.05it/s]


正在保存结果...

处理完成，结果已保存到 F:\TUB\greenlandProject\ERA5LAND\195004allfeature.csv
总处理点数: 86629
新增列: ['sp', 'ssrd', 'str', 'strd', 't2m', 'u10', 'v10', 'asn', 'fal', 'year', 'month']


In [6]:
#-----------------------------提取1950年5-9月所有气象参数到有LLA的文件中-----------并以年月-allfeature命名新文件---------
import pandas as pd
import xarray as xr
import numpy as np
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import os

# 配置参数
BASE_PATH = r"F:\TUB\greenlandProject\ERA5LAND"
CSV_PATH = os.path.join(BASE_PATH, "LaiLonAlt.csv")
NC_PATH = os.path.join(BASE_PATH, "ProcessFile", "ERA5LANDfeature.nc")
OUTPUT_DIR = os.path.join(BASE_PATH, "MonthlyFeatures")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 预加载数据
print("正在加载基础数据...")
df_points = pd.read_csv(CSV_PATH)
ds = xr.open_dataset(NC_PATH)
times = pd.to_datetime(ds['valid_time'].values)
lats = ds.latitude.values
lons = ds.longitude.values

# 创建经纬度查找树（加速最近邻搜索）
def create_lookup_index(array):
    return {val: idx for idx, val in enumerate(array)}

lat_idx_map = create_lookup_index(lats)
lon_idx_map = create_lookup_index(lons)

def find_nearest_index(array, value):
    idx = np.abs(array - value).argmin()
    return idx

# 并行处理函数
def process_month(year_month):
    year, month = year_month
    print(f"\n开始处理 {year}年{month}月...")
    
    # 查找时间索引
    target_time = pd.Timestamp(f"{year}-{month:02d}-01")
    time_idx = find_nearest_index(times, target_time)
    if times[time_idx] != target_time:
        print(f"警告: 使用最接近的时间: {times[time_idx]}")
    
    # 预分配结果数组
    result = np.empty((len(df_points), 9), dtype=np.float32)
    
    # 获取当月数据块
    data_block = ds.isel(valid_time=time_idx)
    
    # 并行处理每个点
    def process_point(i):
        lat = df_points.at[i, 'latitude']
        lon = df_points.at[i, 'longitude']
        
        # 使用预计算的索引加速
        lat_idx = find_nearest_index(lats, lat)
        lon_idx = find_nearest_index(lons, lon)
        
        # 提取数据
        result[i, 0] = data_block['sp'].values[lat_idx, lon_idx]
        result[i, 1] = data_block['ssrd'].values[lat_idx, lon_idx]
        result[i, 2] = data_block['str'].values[lat_idx, lon_idx]
        result[i, 3] = data_block['strd'].values[lat_idx, lon_idx]
        result[i, 4] = data_block['t2m'].values[lat_idx, lon_idx]
        result[i, 5] = data_block['u10'].values[lat_idx, lon_idx]
        result[i, 6] = data_block['v10'].values[lat_idx, lon_idx]
        result[i, 7] = data_block['asn'].values[lat_idx, lon_idx]
        result[i, 8] = data_block['fal'].values[lat_idx, lon_idx]
    
    # 使用线程池并行处理
    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(process_point, range(len(df_points))), total=len(df_points), desc=f"处理{year}{month:02d}"))
    
    # 构建结果DataFrame
    result_df = df_points.copy()
    var_names = ['sp', 'ssrd', 'str', 'strd', 't2m', 'u10', 'v10', 'asn', 'fal']
    for i, var in enumerate(var_names):
        result_df[var] = result[:, i]
    
    result_df['year'] = year
    result_df['month'] = month
    
    # 保存结果
    output_path = os.path.join(OUTPUT_DIR, f"{year}{month:02d}allfeature.csv")
    result_df.to_csv(output_path, index=False)
    print(f"{year}年{month}月数据已保存到 {output_path}")
    return output_path

# 要处理的年月列表
months_to_process = [
    (1950, 5), (1950, 6), (1950, 7), 
    (1950, 8), (1950, 9)
]

# 批量处理所有月份
print(f"\n开始批量处理 {len(months_to_process)} 个月份数据...")
for year_month in months_to_process:
    process_month(year_month)

print("\n所有月份处理完成！")

正在加载基础数据...

开始批量处理 5 个月份数据...

开始处理 1950年5月...


处理195005: 100%|██████████| 86629/86629 [00:59<00:00, 1448.38it/s]


1950年5月数据已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195005allfeature.csv

开始处理 1950年6月...


处理195006: 100%|██████████| 86629/86629 [01:00<00:00, 1438.68it/s]


1950年6月数据已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195006allfeature.csv

开始处理 1950年7月...


...

1950年7月数据已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195007allfeature.csv

开始处理 1950年8月...


处理195008: 100%|██████████| 86629/86629 [01:00<00:00, 1431.78it/s]


1950年8月数据已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195008allfeature.csv

开始处理 1950年9月...


处理195009: 100%|██████████| 86629/86629 [01:00<00:00, 1437.15it/s]


1950年9月数据已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195009allfeature.csv

所有月份处理完成！


In [ ]:
#-----------------------------提取所有年份5-9月所有气象参数到有LLA的文件中-----------并以年月-allfeature命名新文件------只用CPU处理---
import pandas as pd
import xarray as xr
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import os
import warnings
from datetime import datetime

# 配置参数
BASE_PATH = r"F:\TUB\greenlandProject\ERA5LAND"
CSV_PATH = os.path.join(BASE_PATH, "LaiLonAlt.csv")
NC_PATH = os.path.join(BASE_PATH, "ProcessFile", "ERA5LANDfeature.nc")
OUTPUT_DIR = os.path.join(BASE_PATH, "MonthlyFeatures")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 设置日志格式
def log_message(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

# 忽略不必要的警告
warnings.filterwarnings("ignore", category=RuntimeWarning)

# 预加载数据
log_message("正在加载基础数据...")
try:
    df_points = pd.read_csv(CSV_PATH)
    log_message(f"成功加载CSV文件，共{len(df_points)}个点位")
    
    ds = xr.open_dataset(NC_PATH)
    times = pd.to_datetime(ds['valid_time'].values)
    log_message(f"NetCDF文件时间范围: {times.min()} 至 {times.max()}")
    
    # 创建经纬度快速查找索引
    lats = ds.latitude.values
    lons = ds.longitude.values
    lat_idx_map = {val: idx for idx, val in enumerate(lats)}
    lon_idx_map = {val: idx for idx, val in enumerate(lons)}
    
except Exception as e:
    log_message(f"数据加载失败: {str(e)}")
    raise

def find_nearest_index(array, value):
    """快速查找最近索引"""
    return np.abs(array - value).argmin()

def process_month(year_month):
    """处理单个月份的核心函数"""
    year, month = year_month
    output_path = os.path.join(OUTPUT_DIR, f"{year}{month:02d}allfeature.csv")
    
    # 检查文件是否已存在
    if os.path.exists(output_path):
        log_message(f"{year}年{month}月数据已存在，跳过处理")
        return None
    
    start_time = datetime.now()
    log_message(f"开始处理 {year}年{month}月数据...")
    
    try:
        # 查找时间索引
        target_time = pd.Timestamp(f"{year}-{month:02d}-01")
        time_idx = find_nearest_index(times, target_time)
        actual_time = times[time_idx]
        
        if actual_time.month != month or actual_time.year != year:
            log_message(f"警告: {year}年{month}月无精确数据，使用最接近的 {actual_time}")
            if abs((actual_time - target_time).days) > 31:
                log_message(f"时间差距过大，跳过该月份")
                return None
        
        # 预分配结果数组
        result = np.empty((len(df_points), 9), dtype=np.float32)
        data_block = ds.isel(valid_time=time_idx)
        
        # 并行处理函数
        def process_point(i):
            lat = df_points.at[i, 'latitude']
            lon = df_points.at[i, 'longitude']
            
            try:
                # 快速查找最近的网格点索引
                lat_idx = lat_idx_map.get(lat, find_nearest_index(lats, lat))
                lon_idx = lon_idx_map.get(lon, find_nearest_index(lons, lon))
                
                # 提取所有变量
                result[i, 0] = data_block['sp'].values[lat_idx, lon_idx]
                result[i, 1] = data_block['ssrd'].values[lat_idx, lon_idx]
                result[i, 2] = data_block['str'].values[lat_idx, lon_idx]
                result[i, 3] = data_block['strd'].values[lat_idx, lon_idx]
                result[i, 4] = data_block['t2m'].values[lat_idx, lon_idx]
                result[i, 5] = data_block['u10'].values[lat_idx, lon_idx]
                result[i, 6] = data_block['v10'].values[lat_idx, lon_idx]
                result[i, 7] = data_block['asn'].values[lat_idx, lon_idx]
                result[i, 8] = data_block['fal'].values[lat_idx, lon_idx]
            except Exception:
                result[i, :] = np.nan
        
        # 使用线程池并行处理（不显示进度条）
        with ThreadPoolExecutor(max_workers=min(8, os.cpu_count())) as executor:
            executor.map(process_point, range(len(df_points)))
        
        # 构建结果DataFrame
        result_df = df_points.copy()
        var_names = ['sp', 'ssrd', 'str', 'strd', 't2m', 'u10', 'v10', 'asn', 'fal']
        for i, var in enumerate(var_names):
            result_df[var] = result[:, i]
        
        result_df['year'] = year
        result_df['month'] = month
        
        # 保存结果
        result_df.to_csv(output_path, index=False)
        elapsed = (datetime.now() - start_time).total_seconds()
        log_message(f"完成 {year}年{month}月数据处理，耗时{elapsed:.1f}秒，已保存到 {output_path}")
        return output_path
    
    except Exception as e:
        log_message(f"处理{year}年{month}月时出错: {str(e)}")
        return None

# 生成要处理的所有年月列表 (1951-2024年，每年4-9月)
years = range(1969, 2024)
months = range(4, 10)  # 4-9月
year_months = [(y, m) for y in years for m in months]

log_message(f"\n开始批量处理 {len(year_months)} 个月份数据...")

# 分批处理避免内存溢出
batch_size = 12  # 每次处理12个月
for i in range(0, len(year_months), batch_size):
    batch = year_months[i:i + batch_size]
    log_message(f"\n正在处理批次 {i//batch_size + 1}/{(len(year_months)-1)//batch_size + 1} (共{len(batch)}个月份)")
    
    for ym in batch:
        process_month(ym)
    
    # 每处理完一批次后手动清理内存
    if hasattr(ds, 'close'):
        ds.close()
    ds = xr.open_dataset(NC_PATH)  # 重新打开文件

log_message("\n所有月份处理完成！")

In [28]:
#-----------------------------提取所有年份5-9月所有气象参数到有LLA的文件中-----------并以年月-allfeature命名新文件----------------GPU-----------
# ----------------------------- 初始化环境与库 -----------------------------
import os
import warnings
import pandas as pd
import xarray as xr
import numpy as np
from numba import cuda
import cupy as cp
from datetime import datetime

# 忽略不必要警告
warnings.filterwarnings("ignore", category=RuntimeWarning)

# 配置路径
BASE_PATH = r"F:\TUB\greenlandProject\ERA5LAND"
CSV_PATH = os.path.join(BASE_PATH, "LaiLonAlt.csv")
NC_PATH = os.path.join(BASE_PATH, "ProcessFile", "ERA5LANDfeature.nc")
OUTPUT_DIR = os.path.join(BASE_PATH, "MonthlyFeatures")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def log_message(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

# ----------------------------- GPU 核函数 -----------------------------
@cuda.jit
def extract_features_gpu(sp, ssrd, str_, strd, t2m, u10, v10, asn, fal,
                         lats, lons, point_lats, point_lons, out_arr):
    i = cuda.grid(1)
    if i < point_lats.shape[0]:
        lat = point_lats[i]
        lon = point_lons[i]

        lat_idx = 0
        lon_idx = 0
        min_lat_diff = 1e10
        min_lon_diff = 1e10

        for j in range(lats.shape[0]):
            diff = abs(lats[j] - lat)
            if diff < min_lat_diff:
                min_lat_diff = diff
                lat_idx = j
        for k in range(lons.shape[0]):
            diff = abs(lons[k] - lon)
            if diff < min_lon_diff:
                min_lon_diff = diff
                lon_idx = k

        if (0 <= lat_idx < sp.shape[0]) and (0 <= lon_idx < sp.shape[1]):
            out_arr[i, 0] = sp[lat_idx, lon_idx]
            out_arr[i, 1] = ssrd[lat_idx, lon_idx]
            out_arr[i, 2] = str_[lat_idx, lon_idx]
            out_arr[i, 3] = strd[lat_idx, lon_idx]
            out_arr[i, 4] = t2m[lat_idx, lon_idx]
            out_arr[i, 5] = u10[lat_idx, lon_idx]
            out_arr[i, 6] = v10[lat_idx, lon_idx]
            out_arr[i, 7] = asn[lat_idx, lon_idx]
            out_arr[i, 8] = fal[lat_idx, lon_idx]
        else:
            for j in range(9):
                out_arr[i, j] = cp.nan

# ----------------------------- 主处理函数 -----------------------------
def process_month(year, month, df_points, ds, times, lats, lons):
    output_path = os.path.join(OUTPUT_DIR, f"{year}{month:02d}allfeature.csv")
    if os.path.exists(output_path):
        log_message(f"{year}年{month}月数据已存在，跳过")
        return

    try:
        log_message(f"开始处理 {year}年{month}月数据...")

        # 找最近的时间索引
        target_time = pd.Timestamp(f"{year}-{month:02d}-01")
        time_idx = np.abs(times - target_time).argmin()
        actual_time = times[time_idx]

        if actual_time.month != month or actual_time.year != year:
            log_message(f"警告: {year}年{month}月无精确数据，使用最接近的 {actual_time}")
            if abs((actual_time - target_time).days) > 31:
                log_message("时间差距过大，跳过该月份")
                return

        data_block = ds.isel(valid_time=time_idx)
        n_points = len(df_points)
        result_gpu = cp.zeros((n_points, 9), dtype=cp.float32)

        # 启动 GPU 核函数
        threads_per_block = 128
        blocks_per_grid = (n_points + threads_per_block - 1) // threads_per_block

        extract_features_gpu[blocks_per_grid, threads_per_block](
            cp.array(data_block['sp'].values),
            cp.array(data_block['ssrd'].values),
            cp.array(data_block['str'].values),
            cp.array(data_block['strd'].values),
            cp.array(data_block['t2m'].values),
            cp.array(data_block['u10'].values),
            cp.array(data_block['v10'].values),
            cp.array(data_block['asn'].values),
            cp.array(data_block['fal'].values),
            cp.array(lats),
            cp.array(lons),
            cp.array(df_points['latitude'].values),
            cp.array(df_points['longitude'].values),
            result_gpu
        )
        cuda.synchronize()

        # 拷贝数据回 CPU 并写出
        result = cp.asnumpy(result_gpu)
        result_df = df_points.copy()
        var_names = ['sp', 'ssrd', 'str', 'strd', 't2m', 'u10', 'v10', 'asn', 'fal']
        for i, var in enumerate(var_names):
            result_df[var] = result[:, i]

        result_df['year'] = year
        result_df['month'] = month

        result_df.to_csv(output_path, index=False)
        log_message(f"完成 {year}年{month}月，保存行数: {len(result_df)} 行，已保存到 {output_path}")

    except Exception as e:
        log_message(f"处理出错: {e}")

# ----------------------------- 主程序入口 -----------------------------
def main():
    log_message("加载基础数据...")
    df_points = pd.read_csv(CSV_PATH)
    ds = xr.open_dataset(NC_PATH)
    times = pd.to_datetime(ds['valid_time'].values)
    lats = ds.latitude.values
    lons = ds.longitude.values
    log_message(f"成功加载，共 {len(df_points)} 个点位，时间范围: {times.min()} 至 {times.max()}")

    # 要处理的-------------------年------------------------------月范围（1969-2023，4-9月）
    years = range(1951, 1952)
    months = range(4, 10)
    year_months = [(y, m) for y in years for m in months]

    batch_size = 12
    for i in range(0, len(year_months), batch_size):
        batch = year_months[i:i+batch_size]
        log_message(f"\n处理批次 {(i//batch_size)+1}/{(len(year_months)-1)//batch_size+1}，共{len(batch)}个月")

        for (y, m) in batch:
            process_month(y, m, df_points, ds, times, lats, lons)

        # 重载 Dataset 以释放内存
        ds.close()
        ds = xr.open_dataset(NC_PATH)

    log_message("\n全部月份处理完成！")

# ----------------------------- 执行主函数 -----------------------------
if __name__ == "__main__":
    main()


[2025-05-08 22:07:45] 加载基础数据...
[2025-05-08 22:07:45] 成功加载，共 86629 个点位，时间范围: 1950-01-01 00:00:00 至 2025-02-01 00:00:00
[2025-05-08 22:07:45] 
处理批次 1/1，共6个月
[2025-05-08 22:07:45] 1951年4月数据已存在，跳过
[2025-05-08 22:07:45] 1951年5月数据已存在，跳过
[2025-05-08 22:07:45] 1951年6月数据已存在，跳过
[2025-05-08 22:07:45] 开始处理 1951年7月数据...
[2025-05-08 22:08:00] 完成 1951年7月，保存行数: 86629 行，已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195107allfeature.csv
[2025-05-08 22:08:00] 开始处理 1951年8月数据...
[2025-05-08 22:08:07] 完成 1951年8月，保存行数: 86629 行，已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195108allfeature.csv
[2025-05-08 22:08:07] 开始处理 1951年9月数据...
[2025-05-08 22:08:14] 完成 1951年9月，保存行数: 86629 行，已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195109allfeature.csv
[2025-05-08 22:08:14] 
全部月份处理完成！
